## Importing Libraries

In [67]:
from ollama import chat
import glob
from tqdm import tqdm
import os
import json
import re

## Setting up files

In [68]:
GENERATION_MODEL = "qwen3:1.7b"

FILES = glob.glob("../Test_Files/Clinical_diaries/clinical-diary*.txt")
GOLD_FILES = glob.glob("../Test_Files/Clinical_diaries/GT-clinical-diary*.json")
SCHEMA = "../Test_Files/Schemas/parameter-extraction_schema.json"

PROMPT_FILE = "./prompts/parameter-extraction_prompt.txt"

OUTPUT_DIR = "./llm-outputs/parameter-extraction/"
OUTPUT_FILE = "experiment"


print(f"Found the following diaries {FILES}")
print(f"Found the following golden diaries {GOLD_FILES}")

Found the following diaries ['../Test_Files/Clinical_diaries\\clinical-diary_e1.txt']
Found the following golden diaries ['../Test_Files/Clinical_diaries\\GT-clinical-diary_e1.json']


## Pre-processing

Removal of unnecessary things from the file

In [51]:
pass

## Parameter Extraction

In this phase the parameters enforced by our client will be extracted from the unstructured clinical diary through a LLM approach

In [52]:
## Setting evironment
with open(PROMPT_FILE,"r", encoding="utf-8") as p:
    prompt_arr = [t.strip() for t in p.readlines() if t.strip()]
    base_prompt = " ".join(prompt_arr)

os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isfile(os.path.join(OUTPUT_DIR, path)):
        count += 1

In [ ]:
pbar = tqdm(total=len(FILES), desc="Processing diaries")

for file in FILES:
    with open(file,"r", encoding="utf-8") as f:
        text_arr = [t.strip() for t in f.readlines() if t.strip()]
        text = " ".join(text_arr)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt.replace("{{DIARY_TEXT}}",text)
        
        stream = chat(
            model=GENERATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            )
        
        llm_output = ""
        for chunk in stream:
            llm_output += chunk["message"]["content"]

        with open(f"{OUTPUT_DIR}{OUTPUT_FILE}-{count}.txt","a",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_FILE}-{count}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing diaries:   0%|          | 0/1 [00:00<?, ?it/s]

processing file: ../Test_Files\clinical-diary_e1.txt


Processing diaries: 100%|██████████| 1/1 [00:59<00:00, 59.24s/it]

Saved LLM output on experiment-0




## Evaluation
In this phase the pipeline of extraction will be evaluated in 3 different fields:
- Field-level accuracy
- Missing field rate
- Schema compliance rate 

In [ ]:
for gold_file in GOLD_FILES:
    with open(gold_file,"r",encoding="utf-8") as gf, \
         open(SCHEMA,"r",encoding="utf-8") as sch, \
         open(f"{OUTPUT_DIR}{OUTPUT_FILE}-{count}.txt","r",encoding="utf-8") as out:

        curr_gf_diary = gold_file.split('_')[3].split('.')[0]

        data_gf = json.load(gf)
        data_sch = json.load(sch)

        out_arr = [t.strip() for t in out.readlines() if t.strip()]
        out_text = " ".join(out_arr)

        outputs = out_text.split("Ouput for file ")
        outputs.pop(0)

        for output in outputs:
            if curr_gf_diary not in output:
                continue

            # Extract JSON safely
            json_match = re.search(r"\{.*\}", output, flags=re.DOTALL)
            if not json_match:
                print("No JSON found for", curr_gf_diary)
                continue

            output_json = json.loads(json_match.group(0))

            print("Golden truth data ", data_gf)
            print("Output of the LLM ", output_json)

            # Schema compliance
            gt_keys = set(data_sch.keys())
            out_keys = set(output_json.keys())
            matched_keys = gt_keys & out_keys

            print(f"The output complied with {len(matched_keys)} out of {len(gt_keys)}, "
                  f"so we have a schema compliance rate of {(len(matched_keys)/len(gt_keys))*100}%")

            # Missing field rate
            out_num_missing = 0
            for key, value in data_gf.items():
                if value is not None:
                    if key not in output_json or output_json[key] in [None, "null", ""]:
                        out_num_missing += 1

            print(f"The output could identify {out_num_missing} fields from the golden truth diary, "
                  f"so missing field rate is {(out_num_missing/len(data_gf))*100}%")

            # Field-level accuracy
            
            def normalize(x):
                if isinstance(x, str) and x.isdigit():
                    return int(x)
                return x

            out_num_right = 0
            for key in data_gf:
                if key in output_json and normalize(data_gf[key]) == normalize(output_json[key]):
                    out_num_right += 1

            print(f"Field-level accuracy: {out_num_right} out of {len(data_gf)} fields correctly identified, so we have a field-level accuracy of {(out_num_right/len(data_gf))*100}%")

Golden truth data  {'age_or_birthdate': 62, 'ecog_ps': 1, 'diagnosis': 'Adenocarcinoma do pulmão direito', 'diagnosis_date': None, 'molecular_status': {'PD-L1': 'TPS 10%', 'EGFR': 'mutação exon 19 (deleção)', 'ALK': 'negativo', 'ROS1': 'negativo'}, 'stage': 'IV A (metastização óssea)', 'treatments': {'1_treatment': {'start_date': '12-11-2025', 'end_date': None}, '2_treatment': {'start_date': None, 'end_date': None}}, 'control': {'brain_metastases': False, 'bone_metastases': True, 'targeted_therapy': 'Osimertinib 80 mg id', 'comorbidities': ['Hipertensão arterial', 'Diabetes mellitus tipo 2', 'Dislipidemia'], 'prior_surgeries': ['Colecistectomia laparoscópica (2018)']}}
Output of the LLM  {'age_or_birthdate': '62', 'ecog_ps': '1', 'diagnosis': 'Adenocarcinoma do pulmão direito', 'diagnosis_date': None, 'molecular_status': 'PD-L1: TPS 10%, EGFR exon 19 deletion, ALK neg, ROS1 neg', 'stage': 'IV A', 'treatments': {'1_treatment': {'start_date': '12-11-2025', 'end_date': None}, '2_treatment